# A.6: Padding and embedding

Pad a batch on the right and on the left, then look up embeddings.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 tiktoken basics
import tiktoken

# From A.2 Loading and training Hugging Face tokenizers
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")

enc_out = tok("Hello world")

print(enc_out["input_ids"], tok.decode(enc_out["input_ids"]), tok.is_fast)

# From A.5 Packing a corpus into training blocks
import numpy as np

import torch

enc = tiktoken.get_encoding("gpt2")

docs = ["First document.", "Second, slightly longer document.", "Third."]

stream = []

for d in docs:
    stream.extend(enc.encode_ordinary(d))
    stream.append(enc.eot_token)                 # <|endoftext|>, ID 50256

print(len(stream), stream)

np.array(stream, dtype=np.uint16).tofile("corpus.bin")   # GPT-2 IDs fit in 16 bits

data = torch.from_numpy(np.fromfile("corpus.bin", dtype=np.uint16).astype(np.int64))

def get_batch(data, batch_size, block_size, generator=None):
    starts = torch.randint(len(data) - block_size, (batch_size,), generator=generator)
    x = torch.stack([data[s:s + block_size] for s in starts])
    y = torch.stack([data[s + 1:s + block_size + 1] for s in starts])
    return x, y

g = torch.Generator().manual_seed(0)

x, y = get_batch(data, batch_size=2, block_size=8, generator=g)

print(x)

print(y)

print(repr(enc.decode(x[0].tolist())), "->", repr(enc.decode(y[0].tolist())))


In [ ]:
tok.pad_token = tok.eos_token
prompts = ["Hello there", "A much longer prompt than the first"]
for side in ["right", "left"]:
    tok.padding_side = side
    batch = tok(prompts, padding=True, return_tensors="pt")
    print(side, batch["input_ids"].tolist(), batch["attention_mask"].tolist())


**Expected output**

Output:

```text
right [[15496, 612, 50256, 50256, 50256, 50256, 50256], [32, 881, 2392, 6152, 621, 262, 717]] [[1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1]]
left [[50256, 50256, 50256, 50256, 50256, 15496, 612], [32, 881, 2392, 6152, 621, 262, 717]] [[0, 0, 0, 0, 0, 1, 1], [1, 1, 1, 1, 1, 1, 1]]
```


In [ ]:
torch.manual_seed(0)
emb = torch.nn.Embedding(num_embeddings=enc.n_vocab, embedding_dim=64)
h = emb(x)
print(emb.weight.shape, h.shape)


**Expected output**

Output:

```text
torch.Size([50257, 64]) torch.Size([2, 8, 64])
```
